# 🟢 Cell 1: 환경 설정 및 동기화
드라이브 마운트, 깃허브 최신화, 라이브러리 설치 및 허깅페이스 로그인을 담당

In [ ]:
import os
import sys
from google.colab import drive, userdata
from huggingface_hub import login

# 1. 구글 드라이브 마운트
drive.mount('/content/drive')

# 2. 깃허브 동기화 및 브랜치 체크아웃
REPO_NAME = "MultiLexNorm_HW11"
GITHUB_TOKEN = userdata.get('GITHUB_TOKEN')
GITHUB_USERNAME = "jaeiko"
GIT_REPO_URL = f"https://{GITHUB_TOKEN}@github.com/{GITHUB_USERNAME}/{REPO_NAME}.git"

print("\n--- 깃허브 저장소 동기화 ---")
if not os.path.exists(f'/content/{REPO_NAME}'):
    !git clone {GIT_REPO_URL}
    print(f"[System] {REPO_NAME} 클론 완료.")
    !cd /content/{REPO_NAME} && git checkout detection-model
else:
    print(f"[System] {REPO_NAME} 폴더 존재. detection-model 최신 코드를 pull 합니다.")
    !cd /content/{REPO_NAME} && git checkout detection-model && git pull origin detection-model

# 3. 작업 디렉토리 이동 및 경로 고정
repo_path = f'/content/{REPO_NAME}'
os.chdir(repo_path)
if repo_path not in sys.path:
    sys.path.insert(0, repo_path)

# 4. 필수 라이브러리 설치 및 호환성 에러 해결
with open('requirements.txt', 'r') as f:
    reqs = f.read()
reqs = reqs.replace('pandas<2.0', 'pandas').replace('numpy<2.0', 'numpy').replace('pyarrow==14.0.2', 'pyarrow')
with open('requirements.txt', 'w') as f:
    f.write(reqs)

!pip install -q -r requirements.txt
!pip install -q bitsandbytes accelerate

# 허깅페이스 모델 다운로드 경로를 구글 드라이브 내부로 강제 지정
os.environ['HF_HOME'] = '/content/drive/MyDrive/HF_CACHE'

# 5. 허깅페이스 로그인
hf_token = userdata.get('HF_TOKEN')
login(token=hf_token)
print("[System] 환경 설정 및 허깅페이스 로그인 완료!")

# Cell 2: Load prompt_mfr_dictionary pipeline


In [ ]:
import glob
import ast
import json
import warnings
import pandas as pd
from tqdm.auto import tqdm
import importlib

# Reload the latest modules pulled from GitHub.
import llm
import pipeline as pipeline_module
import prompt_mfr_adapter
importlib.reload(llm)
importlib.reload(prompt_mfr_adapter)
importlib.reload(pipeline_module)

from llm import MultilingualCorrector
from detection import AnomalyDetector
from pipeline import LexicalNormalizationPipeline
from prompt_mfr_adapter import PromptMFRResources

warnings.filterwarnings('ignore')


def load_gemma_sentence_predictions(flat_pred_path: str) -> list[int]:
    """Read prediction.txt as sentence-level 0/1 flags aligned to validation rows."""
    with open(flat_pred_path, 'r', encoding='utf-8') as f:
        return [int(line.strip()) for line in f if line.strip() in ['0', '1']]

print("[System] prompt_mfr_dictionary pipeline modules loaded.")


# Cell 3: Run prompt_mfr_dictionary full validation


In [ ]:
import glob
import ast
import json
from pathlib import Path
import pandas as pd
from tqdm.auto import tqdm

print("\n--- prompt_mfr_dictionary full validation pipeline ---")

# 1. Build models and resources.
model_path = "/content/drive/MyDrive/MultiLexNorm2026/models/xlmr_finetuned_colab"
test_detector = AnomalyDetector(model_path)

# Load the prompt + MFR dictionary package through the adapter.
prompt_mfr_dir = "/content/MultiLexNorm_HW11/prompt_mfr_dictionary"
resources = PromptMFRResources(prompt_mfr_dir)

# LLM is used only for target normalization.
# prediction.txt and XLM-R provide the detection signal, so target detection prompts are disabled.
test_llm = MultilingualCorrector("Qwen/Qwen2.5-7B-Instruct")

pipeline = LexicalNormalizationPipeline(
    detector=test_detector,
    dictionary=None,
    llm=test_llm,
    prompt_mfr_resources=resources,
    use_prompt_mfr_dictionary=True,
    run_target_detection=False,
)

# 2. Load validation data and sentence-level LLM detector prediction.txt.
prediction_pattern = "/content/MultiLexNorm_HW11/*/LLM-base*/experiments/exp_009_retrieval_fewshot_v1_full_all_k20/predictions.txt"
found_prediction_paths = glob.glob(prediction_pattern)
if not found_prediction_paths:
    raise FileNotFoundError(f"prediction.txt not found: {prediction_pattern}")

prediction_txt_path = found_prediction_paths[0]
gemma_flags_list = load_gemma_sentence_predictions(prediction_txt_path)
print(f"[System] prediction.txt: {prediction_txt_path}")
print(f"[System] prediction rows: {len(gemma_flags_list)}")

val_path = "/content/MultiLexNorm_HW11/multilexnorm2026-dataset/data/validation-00000-of-00001.parquet"
val_df = pd.read_parquet(val_path)

total_samples = len(val_df)
if len(gemma_flags_list) < total_samples:
    raise ValueError(
        f"prediction.txt has {len(gemma_flags_list)} rows, but validation has {total_samples} rows."
    )

print(f"[System] Running full validation inference: {total_samples} sentences")

# 3. Batch inference and online token accuracy.
BATCH_SIZE = 8
final_predictions = []
correct_tokens_count = 0
total_tokens_count = 0

for i in tqdm(range(0, total_samples, BATCH_SIZE), desc="Batch Inferencing"):
    batch_idx = range(i, min(i + BATCH_SIZE, total_samples))

    batch_raw = []
    batch_norm = []
    batch_gemma = []
    batch_langs = []

    for idx in batch_idx:
        raw = val_df['raw'].iloc[idx]
        if isinstance(raw, str):
            raw = ast.literal_eval(raw)
        batch_raw.append([str(t) for t in raw])

        norm = val_df['norm'].iloc[idx]
        if isinstance(norm, str):
            norm = ast.literal_eval(norm)
        batch_norm.append([str(t) for t in norm])

        # Same alignment rule as the original colab_MultiLexNorm_HW11.ipynb:
        # validation row idx uses prediction.txt row idx.
        batch_gemma.append(gemma_flags_list[idx])

        lang_code = val_df['language'].iloc[idx] if 'language' in val_df.columns else val_df['lang'].iloc[idx]
        batch_langs.append(str(lang_code))

    results = pipeline.process_batch(batch_raw, batch_gemma, batch_langs)
    final_predictions.extend(results)

    for res, norm_ans in zip(results, batch_norm):
        total_tokens_count += len(norm_ans)
        correct_tokens_count += sum(1 for p, n in zip(res, norm_ans) if p == n)

    # Backup for Colab runtime interruption.
    if len(final_predictions) % 500 < BATCH_SIZE:
        with open("predictions_full_backup_prompt_mfr.json", "w", encoding="utf-8") as f:
            json.dump(final_predictions, f, ensure_ascii=False)

# 4. Save final predictions.
output_path = "predictions_validation_full_prompt_mfr.json"
with open(output_path, "w", encoding="utf-8") as f:
    json.dump(final_predictions, f, ensure_ascii=False)

accuracy = (correct_tokens_count / total_tokens_count) * 100
print(f"\n[Done] Saved full validation predictions to {output_path}")
print(f"[Metric] Token Accuracy: {accuracy:.2f}%")


In [ ]:
# ERR = (TP - FP) / (TP + FN)
TP = FP = FN = 0
for pred, norm, raw in zip(final_predictions, val_df['norm'], val_df['raw']):
    raw = [str(t) for t in (ast.literal_eval(raw) if isinstance(raw, str) else raw)]
    norm = [str(t) for t in (ast.literal_eval(norm) if isinstance(norm, str) else norm)]
    for p, n, r in zip(pred, norm, raw):
        TP += int(r != n and p == n); FN += int(r != n and p != n); FP += int(r == n and p != r)
ERR = (TP - FP) / (TP + FN) if (TP + FN) else 0
print(f"ERR: {ERR:.4f} ({ERR * 100:.2f}%) | TP={TP}, FP={FP}, FN={FN}")
